# Analysis for scaffold hopping

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm


In [ ]:
from tools import (
    compute_uniqueness,
    compute_novelty,
    compute_unique_novelty,
)

## Load data

In [ ]:
pred_dir = Path("predictions/conditional_mol/")
files = sorted(f for f in pred_dir.rglob("*.csv") if not f.stem.endswith("conditional"))
dfs = []
for file in files:
    print(file)
    df = pd.read_csv(file, low_memory=False).reset_index(drop=True)
    # Evaluation script does not break molecules correctly, introducting these rows
    df = df[~df["fail"].fillna(0).astype(bool)].reset_index(drop=True)
    df = df.drop(columns=["index", "fail"], errors="ignore")

    df_cond = pd.read_csv(Path(file).parent / (Path(file).stem + "_conditional.csv"), low_memory=False).reset_index(drop=True)
    df_cond.columns = [c.lower().replace(" ", "_") for c in df_cond.columns]
    df_cond = df_cond.drop(columns=["index", "fail", "error"], errors="ignore")

    if len(df) != len(df_cond):
        print(f"Lengths different: {len(df)} != {len(df_cond)}")
        continue
    df = pd.concat([df, df_cond], axis=1, ignore_index=False)

    attempt = int(Path(file).parent.stem.split("_")[1])
    df["table"] = "time" if attempt == 1 else "var"
    df["method"] = Path(file).stem
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

In [ ]:
# load references
truth = pd.read_csv("/homes/buttensc/Projects/semla-flow/data/conditional_mol/test_first_1000.csv")
truth = truth[truth.fail != 1.0]
reference_smiles = set(truth["smiles"].values)

df_train = pd.read_csv("data/unconditional/geom-drugs/train.csv")
reference_smiles = set(df_train["smiles"].values)

print(len(reference_smiles))

In [ ]:
# enrichment
df["total_number"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]
df["novel"] = df["smiles"].map(lambda x: x not in reference_smiles)
df["valid_novel"] = df["valid"] & df["novel"]
df["valid_smiles"] = df["valid"].astype(bool) * df["smiles"]
df["valid_scaffold_rdkit_csk"] = df["valid"].astype(bool) & df["scaffold_conserved"].astype(bool)
df["valid_scaffold_hop_smiles"] = (~df["valid_scaffold_rdkit_csk"]) * df["valid_smiles"]

In [ ]:
df["integration steps"] = df["method"].str.split("_").str[1].str[1:].astype(int)
df["integration time"] = df["integration steps"] / 100
df["sigma"] = df["method"].str.split("_").str[2].str[1:].astype(float)

In [ ]:
aggs = {
    "tanimoto": ("tanimoto", "max"),
    "shape_sim_1": ("shape_sim_1", "max"),
    "shape_sim_2": ("shape_sim_2", "max"),
    "esp_sim_1": ("esp_sim_1", "max"),
    "esp_sim_2": ("esp_sim_2", "max"),
    "feat_map_score_1": ("feat_map_score_1", "max"),
    "feat_map_score_2": ("feat_map_score_2", "max"),
    "sucos_1": ("sucos_1", "max"),
    "sucos_2": ("sucos_2", "max"),
    "scaffold_conserved": ("scaffold_conserved", "max"),
}
df_best = (
    df[df.valid & df.novel]
    .groupby(
        [
            "table",
            "method",
            "integration steps",
            "integration time",
            "sigma",
            "reference_molecule",
            # "smiles_cond",
            "smiles_pred",
            # "scaffold_cond",
            "scaffold_pred",
        ],
        observed=True,
    )
    .agg(**aggs)
    .reset_index()
)
df_best

## Tables

In [ ]:
def wrap_envs(latex: str, envs: list[str] = ["center", "small", "sc"]) -> str:
    for env in envs:
        latex = latex.replace(r"\begin{tabular}", r"\begin{" + env + "}" + "\n" + r"\begin{tabular}")
        latex = latex.replace(r"\end{tabular}", r"\end{tabular}" + "\n" + r"\end{" + env + "}")
    return latex


In [ ]:
df.columns

### Validity

In [ ]:
def mean(x):
    n = 100000
    if len(x) == 0:
        return float("nan")
    if len(x) > n:
        return np.mean(x)
    return np.sum(x) / n


aggs = {
    "Total number": ("total_number", "sum"),
    r"Generated \unit{\percent}": ("total_number", mean),
    r"Connected \unit{\percent}": ("connected", mean),
    r"Chemical \unit{\percent}": ("chemical", mean),
    r"Physical \unit{\percent}": ("physical", mean),
    r"Valid \unit{\percent}": ("valid", mean),
    # "valid_scaffold_hop": ("scaffold_rdkit_csk", 1 - mean),
}
df_agg = df[df.table == "time"].groupby(["integration time", "sigma"], observed=False).agg(**aggs)
df_agg.index.names = ["\(t\)", "\(\sigma\)"]
cols = df_agg.columns
df_style = df_agg.style.format_index("{:.2f}").format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:])
df_style

In [ ]:
caption = """
Validity of the on a reference molecule conditioned generated molecules for \(\sigma = 0.1 \).
The table contains the total number of molecules that was to be generated and the proportions of molecules requiring connectedness, chemical validity, physical validity, and overall validity.
As defined in the method, a molecule with a conformation is valid only if it is connected, chemically valid, and physically valid.
"""
label = "tab:cond_mol_validity_time"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    sparse_columns=False,
)
latex = latex.replace("%", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
with open("tables/cond_mol_validity_time.tex", "w") as f:
    f.write(latex)

In [ ]:
df_agg = df[df.table == "var"].groupby(["integration time", "sigma"], observed=False).agg(**aggs)
df_agg.index.names = ["\(t\)", "\(\sigma\)"]
cols = df_agg.columns
df_style = df_agg.style.format_index("{:.2f}").format("{:.0f}", subset=cols[:1]).format("{:.1%}", subset=cols[1:])
df_style

In [ ]:
caption = """
Validity of the on a reference molecule conditioned generated molecules for \( t = 0.72 \).
The table contains the total number of molecules that was to be generated and the proportions of molecules requiring connectedness, chemical validity, physical validity, and overall validity.
As defined in the method, a molecule with a conformation is valid only if it is connected, chemically valid, and physically valid.
"""
label = "tab:cond_mol_validity_var"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    sparse_columns=False,
)
latex = latex.replace("%", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
with open("tables/cond_mol_validity_var.tex", "w") as f:
    f.write(latex)

### Properties

In [ ]:
df_filter = df[df.valid]
aggs = {
    "qed": ["mean", "std"],
    "sa": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
    "weight": ["mean", "std"],
    "num_heavy": ["mean", "std"],
    "num_rings": ["mean", "std"],
    "lipinski": ["mean", "std"],
    "logp": ["mean", "std"],
    "spacial": ["mean", "std"],
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Energy ratio

In [ ]:
df_filter = df[df.valid]
df_filter = df
aggs = {
    "ensemble_avg_energy": ["mean", "std"],
    "mol_pred_energy": ["mean", "std"],
    "energy_ratio": ["mean", "std"],
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(aggs)
cols = df_agg.columns
df_agg.style.format("{:.2f}", subset=cols)

### Novel, Unique, Valid, Scaffold Hop

In [ ]:
n = 100000

df_filter = df[df.table == "time"]
aggs = {
    "Valid": ("valid", lambda x: sum(x) / n),
    # "Valid & Scaffold Hop": ("scaffold_rdkit_csk", lambda x: 1 - mean(x)),
    "Valid \& Novel": (
        "valid_smiles",
        lambda x: compute_novelty(x, reference_smiles, total=1) / n,
    ),
    "Valid \& Unique": (
        "valid_smiles",
        lambda x: compute_uniqueness(x, total=1) / n,
    ),
    "Valid \& Unique \& Novel": (
        "valid_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n,
    ),
    "Valid \& Unique \& Novel \& Scaffold Hop": (
        "valid_scaffold_hop_smiles",
        lambda x: compute_unique_novelty(x, reference_smiles, total=1) / n,
    ),
}
df_agg = df_filter.groupby(["integration time", "sigma"], observed=False).agg(**aggs)
df_agg.index.names = ["\(t\)", "\(\sigma\)"]
cols = df_agg.columns
df_style = df_agg.style.format_index("{:.2f}").format("{:.2%}")
df_style


In [ ]:
caption = """
Validity, uniqueness, novelty, and share of scaffold hops of the on a reference molecule conditioned generated molecules for \( \sigma = 0.10 \).
These are proportions of the total number of molecules that were to be generated.
"""
label = "tab:cond_mol_novelty_time"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    sparse_columns=False,
)
latex = latex.replace("%", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
with open("tables/cond_mol_novelty_time.tex", "w") as f:
    f.write(latex)

In [ ]:
n = 100000

df_filter = df[df.table == "var"]
df_agg = df_filter.groupby(["integration time", "sigma"], observed=False).agg(**aggs)
df_agg.index.names = ["\(t\)", "\(\sigma\)"]
cols = df_agg.columns
df_style = df_agg.style.format_index("{:.2f}").format("{:.2%}")
df_style

In [ ]:
caption = """
Validity, uniqueness, novelty, and share of scaffold hops of the on a reference molecule conditioned generated molecules for \( t = 0.72 \).
These are proportions of the total number of molecules that were to be generated.
"""
label = "tab:cond_mol_novelty_var"
latex = df_style.to_latex(
    label=label,
    siunitx=True,
    environment="table*",
    hrules=True,
    caption=caption,
    sparse_columns=False,
)
latex = latex.replace("%", "").replace("nan", "")
latex = wrap_envs(latex, ["center"])
with open("tables/cond_mol_novelty_var.tex", "w") as f:
    f.write(latex)

### Similarity

In [ ]:
threshold_tanimoto = 0.8
threshold_sucos = 0.55

df_filter = df[df.valid]
df_filter = df
aggs = {
    "tanimoto mean": ("tanimoto", "mean"),
    "tanimoto std": ("tanimoto", "std"),
    f"tanimoto > {threshold_tanimoto}": (
        "sucos",
        lambda x: sum(x > threshold_tanimoto) / n,
    ),
    "sucos mean": ("sucos", "mean"),
    "sucos std": ("sucos", "std"),
    f"sucos > {threshold_sucos}": ("sucos", lambda x: sum(x > threshold_sucos) / n),
}
df_agg = df_filter.groupby(["table", "integration steps", "sigma"], observed=False).agg(**aggs)
df_agg[f"tanimoto > {threshold_tanimoto} and sucos > {threshold_sucos}"] = (
    df_agg[f"sucos > {threshold_sucos}"] * df_agg[f"tanimoto > {threshold_tanimoto}"]
)
cols = df_agg.columns
df_agg.style.format("{:.2%}", subset=cols)

In [ ]:
threshold_sucos = 0.7
aggs = {
    "number_reference_mols": (
        "sucos",
        lambda x: sum(x > threshold_sucos) > 0,
    ),
}
print(f"number of reference molecules for which valid unique molecules with sucos > {threshold_sucos} were generated")
df_agg = df_best.groupby(["table", "integration steps", "sigma", "reference_molecule"]).agg(**aggs)
df_agg.groupby(["table", "integration steps", "sigma"]).sum()

In [ ]:
df_agg = df_best.groupby(["table", "reference_molecule"]).agg(**aggs)
df_agg.groupby(["table"]).sum()

In [ ]:
# sns.scatterplot(data=df_best, x="tanimoto", y="sucos", hue="method", alpha=0.1)

# Plots

In [ ]:
metrics = {
    "sucos": "SuCOS",
    "sucos_1": "SuCOS",
    "sucos_2": "SuCOS",
    "shape_sim": "Shape Similarity",
    "shape_sim_1": "Shape Similarity",
    "shape_sim_2": "Shape Similarity",
    "esp_sim": "Electrostatic Potential Similarity",
    "esp_sim_1": "Electrostatic Potential Similarity",
    "esp_sim_2": "Electrostatic Potential Similarity",
    "feat_map_score": "Feature Map Score",
    "feat_map_score_1": "Feature Map Score",
    "feat_map_score_2": "Feature Map Score",
    "tanimoto": "ECFP4 Bit Tanimoto",
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "QED",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

In [ ]:
metric = "shape_tanimoto"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "time")].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "feat_map_score_1"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "time")].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "esp_sim_1"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "time")].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(-1, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "esp_sim_1"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "var")].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(-1, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "sucos_1"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "time")].groupby(["table", "method", "smiles_pred"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "sucos_1"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "var")].groupby(["table", "method", "smiles_pred"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "sucos_2"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "time")].groupby(["table", "method", "smiles_pred"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "sucos_2"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "var")].groupby(["table", "method", "smiles_pred"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
metric = "sucos_2"
metric_name = metrics[metric]
fig = sns.histplot(
    df_best[(df_best.table == "var") & (df_best.scaffold_conserved)].groupby(["table", "method", "scaffold_pred"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    # fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of the best {metric_name} of Unique Novel Valid Scaffolds",
    xlabel=metric_name,
)

In [ ]:
metric = "shape_sim_1"
metric_name = metrics[metric]
fig = sns.ecdfplot(
    df_best[df_best.table == "var"].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.5, 1),
    title=f"Out of 100,000 molecules generated, how many\n Unique Novel Molecules have {metric_name} larger than x?",
    xlabel=metric_name,
)

In [ ]:
metric = "sucos_1"
metric_name = metrics[metric]
fig = sns.ecdfplot(
    df_best[df_best.table == "var"].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.5, 1),
    title=f"Out of 100,000 molecules generated, how many\n Unique Novel Molecules have {metric_name} larger than x?",
    xlabel=metric_name,
)

In [ ]:
metric = "esp_sim_1"
metric_name = metrics[metric]
fig = sns.ecdfplot(
    df_best[df_best.table == "var"].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.0, 1),
    title=f"Out of 100,000 molecules generated, how many\n Unique Novel Molecules have {metric_name} larger than x?",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?

metric = "sucos_1"
metric_name = metrics[metric]
fig = sns.ecdfplot(
    df_best[df_best.table == "var"].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.5, 1),
    title=f"Out of 100,000 molecules generated, how many\n Unique Novel Molecules have {metric_name} larger than x?",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?

metric = "sucos_1"
metric_name = metrics[metric]
fig = sns.ecdfplot(
    df_best[(df_best.table == "var") & (~df_best.scaffold_conserved)].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    complementary=True,
    # common_norm=False,
    stat="count",
    # element="step",
    # fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0.5, 1),
    title="Out of 100,00 generated molecules, how many Unique Novel\n Valid Molecules have a new scaffold and SuCOS larger than x?",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?

metric = "sucos_1"
metric_name = metrics[metric]

fig = sns.histplot(
    df_best[(df_best.table == "time")].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?

metric = "sucos_1"
metric_name = metrics[metric]

fig = sns.histplot(
    df_best[(df_best.table == "time") & (~df_best.scaffold_conserved)].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    xlim=(0, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Molecules with new Scaffolds",
    xlabel="SuCOS",
)

In [ ]:
# how many interesting new molecules have we created?
metric = "shape_sim"
metric_name = metrics[metric]

fig = sns.histplot(
    df_best[(df_best.table == "var") & (~df_best.scaffold_conserved)].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(-1, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Scaffold Hops",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?
metric = "esp_sim_1"
metric_name = metrics[metric]

fig = sns.histplot(
    df_best[(df_best.table == "var") & (~df_best.scaffold_conserved)].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    cumulative=True,
    fill=False,
    # bins=100,
    bins=10000,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(-1, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Scaffold Hops",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?
metric = "sucos_1"
metric_name = metrics[metric]

fig = sns.histplot(
    df_best[(df_best.table == "var") & (~df_best.scaffold_conserved)].groupby(["table", "method", "smiles"]).agg({metric: "max"}),
    x=metric,
    hue="method",
    # complementary=True,
    # common_norm=False,
    stat="count",
    element="step",
    fill=False,
    bins=100,
    # legend=True, palette="tab10", linewidth=1.5
)
fig.set(
    # xlim=(-1, 1),
    # ylim=(0, 2000),
    title=f"Distribution of {metric_name} of Unique Novel Valid Scaffold Hops",
    xlabel=metric_name,
)

In [ ]:
# how many interesting new molecules have we created?

df_plot = (
    df_best[df_best.table == "time"].groupby(["table", "method", "smiles", "scaffold_conserved"]).agg({"sucos_1": "max"})
).reset_index()
df_plot_all = df_plot.copy()
df_plot_all["subset"] = "All"
df_plot_scaffold_hop = df_plot[~df_plot.scaffold_conserved].copy()
df_plot_scaffold_hop["subset"] = "Scaffold Hop"
df_plot = pd.concat([df_plot_all, df_plot_scaffold_hop], ignore_index=True).reset_index(drop=True)


In [ ]:
def ecdf(s: pd.Series) -> pd.Series:
    sq = s.value_counts()
    return sq.sort_index(ascending=False).cumsum() * 1.0 / sum(sq)

# Appendix

## Old tables

In [ ]:
cols = [
    "table",
    "method",
    "scaffold_conserved",
]
df_scaff = df[df.valid_novel][cols].groupby(["table", "integration steps", "sigma"]).mean()
df_scaff.style.format("{:.2%}")

# Old plots

In [ ]:
metric = "sucos"
# metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[df.valid_novel][df.table == "table 1"][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    # cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
metric = "sucos"
# metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[(df.valid_novel & (df.table == "table 2"))][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    # cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
# metric = "sucos"
metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[(df.valid_novel & (df.table == "table 1"))][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
# metric = "sucos"
metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[(df.valid_novel & (df.table == "table 2"))][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

## Old code 

In [ ]:
# How much repetition is there? How unique are the generated molecules?
s = df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(compute_uniqueness)
s.name = "Uniqueness"
s

In [ ]:
# How many of the valid generated molecules are not in the test set?
s = df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(compute_novelty)
s.name = "Novelty"
s

In [ ]:
# How many of the valid generated molecules are in the test set?
s = (df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(compute_novelty) - 1).abs()
s.name = "In Test Set"
s

In [ ]:
# How many valid, unique and new molecules have we generated?
s = df.groupby(["table", "integration steps", "sigma"])["smiles_pred"].agg(compute_unique_novelty)
s.name = "Unique Novelty"
s